# ES启动

```
su elasticsearch
nohup /usr/share/elasticsearch/bin/elasticsearch &
```

# ES测试

In [1]:
from elasticsearch import Elasticsearch

# Create an Elasticsearch client instance
es = Elasticsearch(
    [{'host': 'localhost', 'port': 9200, 'scheme': 'http'}]
)

# Check if Elasticsearch is running by sending a simple GET request
def check_elasticsearch():
    response = es.ping()
    if response:
        print("Elasticsearch is up and running")
    else:
        print("Elasticsearch is down")

# Indexing data into Elasticsearch
def insert_data(index_name, doc_type, data):
    res = es.index(index=index_name, body=data)
    print("Data inserted:", res['result'])

# Searching data in Elasticsearch
def search_data(index_name, query):
    res = es.search(index=index_name, body=query)
    print("Search results:")
    for hit in res['hits']['hits']:
        print(hit['_source'])

if __name__ == "__main__":
    # Check if Elasticsearch is running
    check_elasticsearch()

    # Example data
    index_name = "my_index"
    doc_type = "my_doc"
    data = {
        "title": "Example document",
        "content": "This is an example document for Elasticsearch",
        "tags": ["elasticsearch", "python"]
    }

    # Insert data into Elasticsearch
    insert_data(index_name, doc_type, data)

    # Define a simple query
    query = {
        "query": {
            "match": {
                "content": "example"
            }
        }
    }

    # Search data in Elasticsearch
    search_data(index_name, query)

Elasticsearch is up and running
Data inserted: created
Search results:
{'title': 'Example document', 'content': 'This is an example document for Elasticsearch', 'tags': ['elasticsearch', 'python']}
{'title': 'Example document', 'content': 'This is an example document for Elasticsearch', 'tags': ['elasticsearch', 'python']}
{'title': 'Example document', 'content': 'This is an example document for Elasticsearch', 'tags': ['elasticsearch', 'python']}
{'title': 'Example document', 'content': 'This is an example document for Elasticsearch', 'tags': ['elasticsearch', 'python']}
{'title': 'Example document', 'content': 'This is an example document for Elasticsearch', 'tags': ['elasticsearch', 'python']}


In [3]:
import requests
import json

# 替换为你的 Elasticsearch 地址
ELASTICSEARCH_URL = "http://127.0.0.1:9200"

# 如果启用了安全认证，请填写用户名和密码
# AUTH = ('elastic', 'your_password')
AUTH = None

# 如果使用 HTTPS 且证书未验证，可以关闭证书验证 (仅用于开发和测试)
VERIFY_SSL = False

In [5]:
ELASTICSEARCH_URL

'http://127.0.0.1:9200'

# 04通过requests库进行ES的_analyze分词

In [6]:
def make_request(method, endpoint, data=None):
    """通用请求函数，处理 URL、认证和异常"""
    url = f"{ELASTICSEARCH_URL}/{endpoint}"
    headers = {'Content-Type': 'application/json'}
    try:
        if data:
            response = requests.request(
                method, url, headers=headers, json=data, auth=AUTH, verify=VERIFY_SSL
            )
        else:
            response = requests.request(
                method, url, headers=headers, auth=AUTH, verify=VERIFY_SSL
            )
        response.raise_for_status()  # 如果请求失败（非2xx），抛出 HTTPError 异常
        return response.json()
    except requests.exceptions.HTTPError as err:
        print(f"HTTP 错误：{err}")
        print(f"响应内容：{err.response.text}")
    except requests.exceptions.RequestException as err:
        print(f"请求失败：{err}")
    return None


def test_connection():
    """测试 Elasticsearch 连接和 Ping"""
    print("--- 正在测试 Elasticsearch 连接 ---")
    response = make_request('GET', '')
    if response:
        print("连接成功！")
        print(json.dumps(response, indent=2, ensure_ascii=False))


def test_common_analyzers():
    """测试常见的 Elasticsearch 内置分词器"""
    print("\n--- 正在测试常见的 Elasticsearch 内置分词器 ---")
    test_text = "Hello, world! This is a test."
    analyzers = ["standard", "simple", "whitespace", "english"]

    for analyzer in analyzers:
        print(f"\n使用分词器：{analyzer}")
        data = {
            "analyzer": analyzer,
            "text": test_text
        }
        response = make_request('POST', '_analyze', data=data)
        if response and 'tokens' in response:
            tokens = [token['token'] for token in response['tokens']]
            print(f"原始文本: '{test_text}'")
            print(f"分词结果: {tokens}")


def test_ik_analyzers():
    """测试 IK 分词器"""
    print("\n--- 正在测试 IK 分词器 ---")
    test_text_zh = "我在使用Elasticsearch，这是我的测试。"
    ik_analyzers = ["ik_smart", "ik_max_word"]

    for analyzer in ik_analyzers:
        print(f"\n使用 IK 分词器：{analyzer}")
        data = {
            "analyzer": analyzer,
            "text": test_text_zh
        }
        response = make_request('POST', '_analyze', data=data)
        if response and 'tokens' in response:
            tokens = [token['token'] for token in response['tokens']]
            print(f"原始文本: '{test_text_zh}'")
            print(f"分词结果: {tokens}")


if __name__ == "__main__":
    # 执行所有测试
    test_connection()
    print("\n" + "=" * 50)
    test_common_analyzers()
    print("\n" + "=" * 50)
    test_ik_analyzers()

--- 正在测试 Elasticsearch 连接 ---
连接成功！
{
  "name": "autodl-container-rml149b1gp-4f704f45",
  "cluster_name": "elasticsearch",
  "cluster_uuid": "3NSy8C2WReeHOc9buFmL9g",
  "version": {
    "number": "8.12.2",
    "build_flavor": "default",
    "build_type": "tar",
    "build_hash": "48a287ab9497e852de30327444b0809e55d46466",
    "build_date": "2024-02-19T10:04:32.774273190Z",
    "build_snapshot": false,
    "lucene_version": "9.9.2",
    "minimum_wire_compatibility_version": "7.17.0",
    "minimum_index_compatibility_version": "7.0.0"
  },
  "tagline": "You Know, for Search"
}


--- 正在测试常见的 Elasticsearch 内置分词器 ---

使用分词器：standard
原始文本: 'Hello, world! This is a test.'
分词结果: ['hello', 'world', 'this', 'is', 'a', 'test']

使用分词器：simple
原始文本: 'Hello, world! This is a test.'
分词结果: ['hello', 'world', 'this', 'is', 'a', 'test']

使用分词器：whitespace
原始文本: 'Hello, world! This is a test.'
分词结果: ['Hello,', 'world!', 'This', 'is', 'a', 'test.']

使用分词器：english
原始文本: 'Hello, world! This is a test.'
分词结果: 

# 05 ES基础

In [8]:
from elasticsearch import Elasticsearch

# 替换为你的 Elasticsearch 地址
ELASTICSEARCH_URL = "http://localhost:9200"

# 如果没有安全认证，直接创建客户端
es_client = Elasticsearch(ELASTICSEARCH_URL)

# 测试连接
if es_client.ping():
    print("连接成功！")
else:
    print("连接失败。请检查 Elasticsearch 服务是否运行。")


连接成功！


In [13]:
# 定义索引名称和映射
index_name = "blog_posts_py"
mapping = {
  "settings": {
    "number_of_shards": 1,#数据分成1片存储
    "number_of_replicas": 0#没有副本
  },
  "mappings": {
    "properties": {
      "title": {
        "type": "text",
        "analyzer": "ik_max_word",
        "search_analyzer": "ik_smart"
      },
      "content": {
        "type": "text",
        "analyzer": "ik_max_word",
        "search_analyzer": "ik_smart"
      },
      "tags": { "type": "keyword" },
      "author": { "type": "keyword" },
      "created_at": { "type": "date" }
    }
  }
}

'''
analyzer 和 search_analyzer 设置为不同分词器意味着：索引时用细粒度分词，搜索时用粗粒度分词。

具体含义
1. analyzer: "ik_max_word"（索引时）
作用：将文本拆分成最细粒度的词语

例子："中华人民共和国" 会被分成：["中华人民共和国", "中华人民", "中华", "华人", "人民共和国", "人民", "共和国"]

目的：尽可能多地匹配可能的搜索词

2. search_analyzer: "ik_smart"（搜索时）
作用：将搜索关键词拆分成最粗粒度的词语

例子：搜索 "中华人民共和国" 只分成：["中华人民共和国"]

目的：提高搜索精准度，避免返回太多不相关的结果


keyword 类型表示字段会被当作一个整体、不可分割的精确值来存储和匹配。比如作者姓名是必须不能被分割的

'''


'\nanalyzer 和 search_analyzer 设置为不同分词器意味着：索引时用细粒度分词，搜索时用粗粒度分词。\n\n具体含义\n1. analyzer: "ik_max_word"（索引时）\n作用：将文本拆分成最细粒度的词语\n\n例子："中华人民共和国" 会被分成：["中华人民共和国", "中华人民", "中华", "华人", "人民共和国", "人民", "共和国"]\n\n目的：尽可能多地匹配可能的搜索词\n\n2. search_analyzer: "ik_smart"（搜索时）\n作用：将搜索关键词拆分成最粗粒度的词语\n\n例子：搜索 "中华人民共和国" 只分成：["中华人民共和国"]\n\n目的：提高搜索精准度，避免返回太多不相关的结果\n\n'

In [11]:
# 检查索引是否存在，如果不存在则创建
if not es_client.indices.exists(index=index_name):
    es_client.indices.create(index=index_name, body=mapping)
    print(f"索引 '{index_name}' 创建成功。")
else:
    print(f"索引 '{index_name}' 已经存在。")

索引 'blog_posts_py' 创建成功。


In [12]:
from datetime import datetime

In [15]:
documents = [
    {
      "title": "Elasticsearch 入门指南",
      "content": "这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。",
      "tags": ["Elasticsearch", "教程", "搜索"],
      "author": "张三",
      "created_at": datetime(2023, 10, 26, 10, 0, 0)
    },
    {
      "title": "深入理解IK分词器",
      "content": "IK分词器是中文分词的优秀工具。它的智能分词和最细粒度分词模式各有优势。",
      "tags": ["分词", "IK", "中文"],
      "author": "李四",
      "created_at": datetime(2023, 10, 25, 15, 30, 0)
    }
]

In [17]:
for doc in documents:
    es_client.index(index=index_name, document=doc)# es_client.index(index=index_name, document=doc) 就是将文档插入（或更新）到指定的索引中。
    print(f"文档已插入: '{doc['title']}'")

# 刷新索引，确保文档可被搜索到
es_client.indices.refresh(index=index_name)

文档已插入: 'Elasticsearch 入门指南'
文档已插入: '深入理解IK分词器'


ObjectApiResponse({'_shards': {'total': 1, 'successful': 1, 'failed': 0}})

In [22]:
# 定义查询函数
def search_docs(query):
    response = es_client.search(index=index_name, body=query)
    print(f"找到 {response['hits']['total']['value']} 条文档：")
    for hit in response['hits']['hits']:
        print(f"得分：{hit['_score']}，文档：{hit['_source']['title']}")

# 1. 查询标题中的 "入门指南"
print("\n--- 1. 查询标题中的 '入门指南' ---")
# dsl 查询逻辑
query_1 = {
  "query": {
    "match": {
      "title": "入门指南"
    }
  }
}
search_docs(query_1)


--- 1. 查询标题中的 '入门指南' ---
找到 1 条文档：
得分：1.6575259，文档：Elasticsearch 入门指南


In [20]:
responses['hits']

{'total': {'value': 1, 'relation': 'eq'},
 'max_score': 1.6575259,
 'hits': [{'_index': 'blog_posts_py',
   '_id': 'twRQvpwBRo4DH1X4qmm8',
   '_score': 1.6575259,
   '_source': {'title': 'Elasticsearch 入门指南',
    'content': '这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。',
    'tags': ['Elasticsearch', '教程', '搜索'],
    'author': '张三',
    'created_at': '2023-10-26T10:00:00'}}]}

In [23]:
# 2. 结合全文和精确匹配查询
print("\n--- 2. 结合全文（搜索技术）和精确匹配（作者：张三） ---")
query_2 = {
  "query": {
    "bool": {
      "must": {
        "match": {
          "content": "搜索技术"
        }
      },
      "filter": {
        "term": {
          "author": "张三"
        }
      }
    }
  }
}
search_docs(query_2)


--- 2. 结合全文（搜索技术）和精确匹配（作者：张三） ---
找到 1 条文档：
得分：1.4113983，文档：Elasticsearch 入门指南


# 06 ES进阶

In [24]:
from elasticsearch import Elasticsearch
import json

In [25]:
def print_search_results(response):
    print(f"找到 {response['hits']['total']['value']} 条文档：")
    for hit in response['hits']['hits']:
        print(f"得分：{hit['_score']}，文档内容：{json.dumps(hit['_source'], ensure_ascii=False, indent=2)}")

In [26]:
es = Elasticsearch("http://localhost:9200")

# 检查连接
if es.ping():
    print("成功连接到 Elasticsearch！")
else:
    print("无法连接到 Elasticsearch，请检查服务是否运行。")

index_name = "products"

成功连接到 Elasticsearch！


In [27]:
# 检查索引是否存在，如果不存在则创建
if not es.indices.exists(index=index_name):
    es.indices.create(
        index=index_name,
        body={
            "mappings": {
                "properties": {
                    "product_id": {"type": "keyword"},
                    "name": {"type": "text", "analyzer": "ik_max_word"},
                    "description": {"type": "text", "analyzer": "ik_smart"},
                    "price": {"type": "float"},
                    "category": {"type": "keyword"},
                    "stock": {"type": "integer"},
                    "on_sale": {"type": "boolean"},
                    "created_at": {"type": "date"}
                }
            }
        }
    )
    print(f"索引 '{index_name}' 创建成功。")
else:
    print(f"索引 '{index_name}' 已存在。")

索引 'products' 创建成功。


In [29]:
# 插入一个新文档
doc_1 = {
    "product_id": "A001",
    "name": "智能手机",
    "description": "最新款智能手机，性能强大，拍照清晰。",
    "price": 4999.50,
    "category": "电子产品",
    "stock": 150,
    "on_sale": True,
    "created_at": "2023-01-15T09:00:00Z"
}
es.index(index=index_name, id="A001", document=doc_1) # 没有保存，就成功了
print("文档 'A001' 已插入。")

文档 'A001' 已插入。


In [31]:
# 插入另一个文档
doc_2 = {
    "product_id": "B002",
    "name": "无线蓝牙耳机",
    "description": "音质卓越，佩戴舒适，超长续航。",
    "price": 699.00,
    "category": "电子产品",
    "stock": 300,
    "on_sale": True,
    "created_at": "2023-02-20T14:30:00Z"
}
es.index(index=index_name, id="B002", document=doc_2)
print("文档 'B002' 已插入。")

文档 'B002' 已插入。


In [33]:
# 刷新索引以确保文档可被搜索到
es.indices.refresh(index=index_name)

# 1. 全文检索（使用 'match' 查询）
# 搜索名称或描述中包含“智能”的商品
print("\n--- 检索 1: 全文检索“智能” ---")
res_1 = es.search(
    index=index_name,
    body={
        "query": {
            "multi_match": {
                "query": "智能",
                "fields": ["name", "description"]
            }
        }
    }
)
print_search_results(res_1)


--- 检索 1: 全文检索“智能” ---
找到 1 条文档：
得分：0.9066489，文档内容：{
  "product_id": "A001",
  "name": "智能手机",
  "description": "最新款智能手机，性能强大，拍照清晰。",
  "price": 4999.5,
  "category": "电子产品",
  "stock": 150,
  "on_sale": true,
  "created_at": "2023-01-15T09:00:00Z"
}


In [38]:
# 2. 结合 'filter' 进行精确过滤
# 搜索价格低于 1000 元且正在促销的电子产品
print("\n--- 检索 2: 结合查询与过滤 ---")
res_2 = es.search(
    index=index_name,
    body={
        "query": {
            "bool": {
                "must": {
                    "match_all": {}  # 匹配所有文档
                },
                "filter": [
                    {"term": {"category": "电子产品"}},
                    {"term": {"on_sale": True}},
                    {"range": {"price": {"lt": 1000}}}
                ]
            }
        }
    }
)
print_search_results(res_2)


--- 检索 2: 结合查询与过滤 ---
找到 1 条文档：
得分：1.0，文档内容：{
  "product_id": "B002",
  "name": "无线蓝牙耳机",
  "description": "音质卓越，佩戴舒适，超长续航。",
  "price": 699.0,
  "category": "电子产品",
  "stock": 300,
  "on_sale": true,
  "created_at": "2023-02-20T14:30:00Z"
}


In [39]:
# 3. 按关键词分组聚合
# 统计不同类别的商品数量
print("\n--- 检索 3: 聚合查询（按类别统计） ---")
res_3 = es.search(
    index=index_name,
    body={
        "aggs": {
            "products_by_category": {
                "terms": {
                    "field": "category",
                    "size": 10
                }
            }
        },
        "size": 0  # 不返回文档结果，只返回聚合结果
    }
)
print_search_results(res_3)


--- 检索 3: 聚合查询（按类别统计） ---
找到 2 条文档：


# 07 ES 向量检索

In [2]:
import json
import time
from elasticsearch import Elasticsearch

In [3]:
es = Elasticsearch("http://localhost:9200")
from sentence_transformers import SentenceTransformer
# 1. 生成向量: 加载预训练模型
# 这个模型可以将句子转换为 768 维的向量
print("正在加载 SentenceTransformer 模型...")
model = SentenceTransformer('/root/.cache/modelscope/hub/BAAI/bge-small-zh-v1.5')
print("模型加载完成。")


ImportError: cannot import name 'cached_download' from 'huggingface_hub' (/root/miniconda3/lib/python3.8/site-packages/huggingface_hub/__init__.py)

In [48]:
torch.__version__

'1.11.0+cu113'

In [46]:
import torch

In [42]:
import os
os.listdir("/root/.cache/modelscope/hub/BAAI/")

['bge-small-zh-v1.5']